# Cypher Fundamentals

## 1. Connect to Neo4j

In [1]:
import os, sys
sys.path.append('..')
from dotenv import load_dotenv
load_dotenv('../.env')

from utils import get_neo4j_client
import pandas as pd

client = get_neo4j_client()

def run(cypher: str, **params) -> pd.DataFrame:
    """Run a Cypher query and show results as a DataFrame."""
    rows = client.query(cypher, params)
    return pd.DataFrame(rows)


## 2. Node Patterns — every variation

A **node pattern** is just parentheses. Inside, you can put:

| Notation | Meaning |
|---|---|
| `()` | Anonymous node — match anything, don't return it |
| `(n)` | A variable `n` bound to any node (any label) |
| `(:Drug)` | Any node with label `Drug`, no variable |
| `(d:Drug)` | A variable `d` bound to a `Drug` node |
| `(d:Drug {name: 'Aspirin'})` | A `Drug` whose `name` property equals `'Aspirin'` |
| `(d:Drug {name: 'Aspirin', drug_class: 'NSAID'})` | Multiple properties (AND) |

In [2]:
# A: anonymous mode, just count everything
run("MATCH (n) RETURN count(n) AS total_nodes")

,total_nodes
0,506


In [3]:
# Form B: bind variable, return label distribution
run("MATCH (n) RETURN labels(n) AS label, count(*) AS n ORDER BY n DESC") 

,label,n
0,[Condition],259
1,[SideEffect],135
2,[Drug],89
3,[DrugClass],23


In [4]:
# Form C: filter by label
run("MATCH (d:Drug) RETURN d.name AS drug ORDER BY drug LIMIT 10")

,drug
0,Alcohol
1,Allopurinol
2,Alprazolam
3,Amiodarone
4,Amlodipine
5,Amoxicillin
6,Apixaban
7,Aspirin
8,Atenolol
9,Atorvastatin


In [5]:
# Form D: inline property filter (single)
run("MATCH (d:Drug {name: 'Aspirin'}) RETURN d.name, d.drug_class, d.description")

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `drug_class` does not exist in database `c621105e`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=51, offset=50>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 50, 'line': 1, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (d:Drug {name: 'Aspirin'}) RETURN d.name, d.drug_class, d.description"


,d.name,d.drug_class,d.description
0,Aspirin,None,Non-steroidal anti-inflammatory drug that irre...


In [6]:
# Form E: inline property filter (multiple = AND)
run("MATCH (d:Drug {drug_class: 'NSAID'}) RETURN d.name AS nsaid ORDER BY nsaid")

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `drug_class` does not exist in database `c621105e`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=16, offset=15>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 15, 'line': 1, 'column': 16}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (d:Drug {drug_class: 'NSAID'}) RETURN d.name AS nsaid ORDER BY nsaid"


""


## Relationship Patterns — directions matter

Relationships in Neo4j are **always directed** when stored, but you can match them in either direction (or both).

| Notation | Meaning | Read it as |
|---|---|---|
| `(a)-[r]->(b)` | Outgoing from `a` to `b` | "a has a relationship pointing to b" |
| `(a)<-[r]-(b)` | Incoming to `a` from `b` | "b has a relationship pointing to a" |
| `(a)-[r]-(b)` | **Either direction** | "a and b are connected" |
| `(a)-->(b)` | Outgoing, no variable / no type | shorthand |
| `(a)-[:TREATS]->(b)` | Specific type, no variable | typed |
| `(a)-[r:TREATS]->(b)` | Specific type, bound to `r` | typed + bound |
| `(a)-[r:TREATS\|CAUSES_SIDE_EFFECT]->(b)` | Either of two types | union of types |

Our schema (from NB01):

```
(Drug)-[:TREATS]->(Condition)
(Drug)-[:CAUSES_SIDE_EFFECT]->(SideEffect)
(Drug)-[:CONTRAINDICATED_FOR]->(Condition)
(Drug)-[:BELONGS_TO_CLASS]->(DrugClass)
(Drug)-[:INTERACTS_WITH]->(Drug)
```

In [7]:
# Outgoing: drugs that treat Pain
run("""
MATCH (d:Drug)-[:TREATS]->(c:Condition {name: 'Pain'})
RETURN d.name AS treats_pain ORDER BY treats_pain
""")

,treats_pain
0,Aspirin
1,Diclofenac
2,Ibuprofen
3,Naproxen


In [8]:
# Incoming: same query, written 'backwards' — identical result
run("""
MATCH (c:Condition {name: 'Pain'})<-[:TREATS]-(d:Drug)
RETURN d.name AS treats_pain ORDER BY treats_pain
""")

,treats_pain
0,Aspirin
1,Diclofenac
2,Ibuprofen
3,Naproxen


In [9]:
# Undirected: 'either direction' — useful for symmetric relationships
# INTERACTS_WITH is stored once but is conceptually symmetric
run("""
MATCH (a:Drug {name: 'Aspirin'})-[:INTERACTS_WITH]-(other:Drug)
RETURN other.name AS interacts_with_aspirin
""")

,interacts_with_aspirin
0,Ibuprofen
1,Rivaroxaban
2,Enoxaparin
3,Clopidogrel
4,Methotrexate
5,Warfarin
6,Heparin
7,Apixaban
8,Ticagrelor
9,Prasugrel


In [10]:
# Multiple types (OR) using the pipe character
run("""
MATCH (d:Drug {name: 'Ibuprofen'})-[r:TREATS|CAUSES_SIDE_EFFECT]->(target)
RETURN type(r) AS rel, labels(target)[0] AS target_type, target.name AS target
ORDER BY rel, target
""")

,rel,target_type,target
0,CAUSES_SIDE_EFFECT,SideEffect,Edema
1,CAUSES_SIDE_EFFECT,SideEffect,GI Bleeding
2,CAUSES_SIDE_EFFECT,SideEffect,GI Upset
3,CAUSES_SIDE_EFFECT,SideEffect,Hypertension
4,CAUSES_SIDE_EFFECT,SideEffect,Renal Impairment
5,TREATS,Condition,Dysmenorrhea
6,TREATS,Condition,Fever
7,TREATS,Condition,Inflammation
8,TREATS,Condition,Osteoarthritis
9,TREATS,Condition,Pain


In [11]:
# Bound relationship variable — inspect properties of `r`
run("""
MATCH (a:Drug)-[r:INTERACTS_WITH]->(b:Drug)
RETURN a.name, b.name, r.severity, r.mechanism
LIMIT 5
""")

,a.name,b.name,r.severity,r.mechanism
0,Aspirin,Ibuprofen,Moderate,Ibuprofen competitively binds to the COX-1 act...
1,Aspirin,Rivaroxaban,Major,Aspirin inhibits platelet aggregation while Ri...
2,Aspirin,Enoxaparin,Major,Additive anticoagulant and antiplatelet effect...
3,Aspirin,Clopidogrel,Major,Both drugs inhibit platelet function through d...
4,Aspirin,Methotrexate,Major,Aspirin reduces renal clearance of Methotrexat...


### Why direction is not the same as 'undirected'

**Why:** the engine matches the *pattern*, not the *edge*. With `-[]-` it tries both directions, so each undirected edge in the data appears twice in the result. Always remember this when counting/aggregating!

In [12]:
# Directed (one way only — as stored)
run("MATCH (:Drug)-[r:INTERACTS_WITH]->(:Drug) RETURN count(r) AS directed_count")

,directed_count
0,244


In [13]:
# Undirected (matches each edge twice — once a→b, once b→a)
run("MATCH (:Drug)-[r:INTERACTS_WITH]-(:Drug) RETURN count(r) AS undirected_count")

,undirected_count
0,488


## `MATCH` vs `OPTIONAL MATCH`

- `MATCH` — if the pattern doesn't exist, **the row disappears**.
- `OPTIONAL MATCH` — if the pattern doesn't exist, return `NULL` for the unmatched parts (like a SQL `LEFT JOIN`).

In [14]:
# Drugs with their side effects — drugs without side effects DISAPPEAR
run("""
MATCH (d:Drug)-[:CAUSES_SIDE_EFFECT]->(s:SideEffect)
RETURN d.name, collect(s.name) AS side_effects
ORDER BY d.name
LIMIT 5
""")

,d.name,side_effects
0,Allopurinol,"[GI Upset, Rash, Elevated Liver Enzymes, Steve..."
1,Alprazolam,"[Sedation, Depression, Dependence, Cognitive I..."
2,Amiodarone,"[Hepatotoxicity, Peripheral Neuropathy, Photos..."
3,Amlodipine,"[Dizziness, Headache, Fatigue, Peripheral Edem..."
4,Amoxicillin,"[Nausea, Rash, Diarrhea, Vomiting, Allergic Re..."


In [15]:
# Drugs with their side effects — drugs without side effects KEPT, with empty list
run("""
MATCH (d:Drug)
OPTIONAL MATCH (d)-[:CAUSES_SIDE_EFFECT]->(s:SideEffect)
RETURN d.name, collect(s.name) AS side_effects
ORDER BY d.name
LIMIT 5
""")

,d.name,side_effects
0,Alcohol,[]
1,Allopurinol,"[GI Upset, Rash, Elevated Liver Enzymes, Steve..."
2,Alprazolam,"[Sedation, Depression, Dependence, Cognitive I..."
3,Amiodarone,"[Hepatotoxicity, Peripheral Neuropathy, Photos..."
4,Amlodipine,"[Dizziness, Headache, Fatigue, Peripheral Edem..."


## Filtering with `WHERE`

`WHERE` is more flexible than the inline `{prop: val}` style — you can use comparisons, lists, string functions, regex, and boolean logic.

In [16]:
# Equality (same as inline form)
run("MATCH (d:Drug) WHERE d.name = 'Aspirin' RETURN d.name, d.drug_class")

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `drug_class` does not exist in database `c621105e`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=58, offset=57>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 57, 'line': 1, 'column': 58}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (d:Drug) WHERE d.name = 'Aspirin' RETURN d.name, d.drug_class"


,d.name,d.drug_class
0,Aspirin,None


In [17]:
# IN — match against a list of values
run("""
MATCH (d:Drug)
WHERE d.name IN ['Aspirin', 'Ibuprofen', 'Warfarin']
RETURN d.name, d.drug_class
""")

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `drug_class` does not exist in database `c621105e`. Verify that the spelling is correct.', position=<SummaryInputPosition line=4, column=18, offset=86>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 86, 'line': 4, 'column': 18}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nMATCH (d:Drug)\nWHERE d.name IN ['Aspirin', 'Ibuprofen', 'Warfarin']\nRETURN d.name, d.drug_class\n"


,d.name,d.drug_class
0,Aspirin,None
1,Ibuprofen,None
2,Warfarin,None


In [18]:
# String functions — case-insensitive partial match
run("""
MATCH (d:Drug)
WHERE toLower(d.name) CONTAINS 'pro'
RETURN d.name
""")

,d.name
0,Ibuprofen
1,Naproxen
2,Metoprolol
3,Propranolol
4,Ciprofloxacin
5,Valproic Acid


In [19]:
# STARTS WITH / ENDS WITH
run("MATCH (d:Drug) WHERE d.name STARTS WITH 'A' RETURN d.name")

,d.name
0,Alcohol
1,Allopurinol
2,Alprazolam
3,Amiodarone
4,Amlodipine
5,Amoxicillin
6,Apixaban
7,Aspirin
8,Atenolol
9,Atorvastatin


In [20]:
# Boolean logic — AND/OR/NOT
run("""
MATCH (d:Drug)
WHERE d.drug_class = 'NSAID' AND NOT d.name = 'Aspirin'
RETURN d.name
""")

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `drug_class` does not exist in database `c621105e`. Verify that the spelling is correct.', position=<SummaryInputPosition line=3, column=9, offset=24>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 24, 'line': 3, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nMATCH (d:Drug)\nWHERE d.drug_class = 'NSAID' AND NOT d.name = 'Aspirin'\nRETURN d.name\n"


""


In [21]:
# Filter on a relationship's existence using pattern in WHERE
run("""
MATCH (d:Drug)
WHERE (d)-[:TREATS]->(:Condition {name: 'Pain'})
RETURN d.name
""")

,d.name
0,Aspirin
1,Ibuprofen
2,Naproxen
3,Diclofenac


In [22]:
# Anti-pattern using NOT — drugs that DO NOT treat Pain
run("""
MATCH (d:Drug)
WHERE NOT (d)-[:TREATS]->(:Condition {name: 'Pain'})
RETURN d.name
LIMIT 10
""")

,d.name
0,Celecoxib
1,Warfarin
2,Heparin
3,Rivaroxaban
4,Apixaban
5,Enoxaparin
6,Dabigatran
7,Clopidogrel
8,Ticagrelor
9,Prasugrel


## `RETURN` — projection, aliasing, ordering, paging

In [23]:
# Aliasing with AS
run("MATCH (d:Drug) RETURN d.name AS drug, d.drug_class AS class LIMIT 5")

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `drug_class` does not exist in database `c621105e`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=41, offset=40>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 40, 'line': 1, 'column': 41}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (d:Drug) RETURN d.name AS drug, d.drug_class AS class LIMIT 5'


,drug,class
0,Aspirin,None
1,Ibuprofen,None
2,Naproxen,None
3,Diclofenac,None
4,Celecoxib,None


In [24]:
# DISTINCT
run("MATCH (d:Drug)-[:BELONGS_TO_CLASS]->(c:DrugClass) RETURN DISTINCT c.name AS drug_class ORDER BY drug_class")

,drug_class
0,ACE Inhibitor
1,ARB
2,Antiarrhythmic
3,Antibiotic
4,Anticoagulant
5,Anticonvulsant
6,Antidiabetic
7,Antiplatelet
8,Benzodiazepine
9,Beta-Blocker


In [25]:
# ORDER BY + SKIP + LIMIT (paging)
run("""
MATCH (d:Drug)
RETURN d.name AS drug
ORDER BY d.name
SKIP 5
LIMIT 5
""")

,drug
0,Amoxicillin
1,Apixaban
2,Aspirin
3,Atenolol
4,Atorvastatin


## Aggregations

When you put an aggregation function in `RETURN`, every other variable becomes a **grouping key** automatically. There is no `GROUP BY` keyword in Cypher.

In [26]:
# count(*) — total rows
run("MATCH (d:Drug) RETURN count(*) AS total_drugs")

,total_drugs
0,89


In [30]:
# Implicit GROUP BY — group by drug_class, count drugs in each
run("""
MATCH (d:Drug)
RETURN d.drug_class AS class, count(*) AS n
ORDER BY n DESC
""")

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `drug_class` does not exist in database `c621105e`. Verify that the spelling is correct.', position=<SummaryInputPosition line=3, column=10, offset=25>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 25, 'line': 3, 'column': 10}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (d:Drug)\nRETURN d.drug_class AS class, count(*) AS n\nORDER BY n DESC\n'


,class,n
0,None,89


In [31]:
# collect() — gather values into a list per group
run("""
MATCH (d:Drug)-[:TREATS]->(c:Condition)
RETURN d.name AS drug, collect(c.name) AS treats
ORDER BY drug
LIMIT 5
""")

,drug,treats
0,Allopurinol,"[Gout, Hyperuricemia, Tumor Lysis Syndrome Pre..."
1,Alprazolam,"[Panic Disorder, Anxiety Disorders]"
2,Amiodarone,"[Atrial Fibrillation, Supraventricular Tachyca..."
3,Amlodipine,"[Hypertension, Angina, Coronary Artery Disease]"
4,Amoxicillin,"[H. pylori Eradication, Upper Respiratory Infe..."


## `WITH` — chaining stages

`WITH` is the bridge between pipeline stages. It works like `RETURN`, but instead of returning to the user, it passes results to the next part of the query. Use it to **filter after aggregation**, like SQL's `HAVING`.

In [32]:
# Find drug classes that have MORE THAN 2 drugs
run("""
MATCH (d:Drug)-[:BELONGS_TO_CLASS]->(c:DrugClass)
WITH c, count(d) AS drug_count
WHERE drug_count > 2
RETURN c.name AS class, drug_count
ORDER BY drug_count DESC
""")

,class,drug_count
0,Antibiotic,7
1,Anticoagulant,6
2,Antidiabetic,6
3,NSAID,5
4,SSRI,5
5,Opioid,5
6,Anticonvulsant,5
7,Statin,4
8,ACE Inhibitor,4
9,Beta-Blocker,4


In [33]:
# Multi-stage: find drugs that treat at least 3 conditions
run("""
MATCH (d:Drug)-[:TREATS]->(c:Condition)
WITH d, count(c) AS n_conditions
WHERE n_conditions >= 3
RETURN d.name AS drug, n_conditions
ORDER BY n_conditions DESC
""")

,drug,n_conditions
0,Ibuprofen,6
1,Naproxen,6
2,Sertraline,6
3,Paroxetine,6
4,Propranolol,6
...,...,...
63,Phenytoin,3
64,Carbamazepine,3
65,Valproic Acid,3
66,Levothyroxine,3


## Variable-length paths

The killer feature of a graph DB. Cypher lets you say *"any number of hops between 1 and N"* with the `[*lo..hi]` syntax.

| Notation | Meaning |
|---|---|
| `[*]` | Any length (use carefully — can explode) |
| `[*1..3]` | Between 1 and 3 hops |
| `[*..3]` | Up to 3 hops |
| `[*2..]` | At least 2 hops |
| `[*2]` | Exactly 2 hops |
| `[:TREATS*1..2]` | 1 or 2 hops, **only** of type `TREATS` |

In [34]:
# Find anything reachable from Aspirin in exactly 2 hops
run("""
MATCH (d:Drug {name: 'Aspirin'})-[*2]-(neighbor)
RETURN DISTINCT labels(neighbor)[0] AS type, neighbor.name AS name
LIMIT 15
""")

,type,name
0,Drug,Enoxaparin
1,Drug,Lisinopril
2,Drug,Enalapril
3,Drug,Losartan
4,Drug,Lithium
5,Drug,Warfarin
6,Drug,Prednisone
7,Drug,Methotrexate
8,SideEffect,GI Bleeding
9,SideEffect,GI Upset


In [35]:
# Bind the entire path to a variable
run("""
MATCH path = (a:Drug {name: 'Aspirin'})-[*1..2]-(b:Drug)
WHERE a <> b
RETURN [n IN nodes(path) | n.name] AS node_chain,
       length(path) AS hops
LIMIT 5
""")

,node_chain,hops
0,"[Aspirin, Ibuprofen]",1
1,"[Aspirin, Ibuprofen, Enoxaparin]",2
2,"[Aspirin, Ibuprofen, Lisinopril]",2
3,"[Aspirin, Ibuprofen, Enalapril]",2
4,"[Aspirin, Ibuprofen, Losartan]",2


In [36]:
# Inspect relationship types along the path
run("""
MATCH path = (a:Drug {name: 'Aspirin'})-[*1..2]-(b)
RETURN [rel IN relationships(path) | type(rel)] AS rel_chain,
       b.name AS target
LIMIT 5
""")

,rel_chain,target
0,[INTERACTS_WITH],Ibuprofen
1,"[INTERACTS_WITH, INTERACTS_WITH]",Enoxaparin
2,"[INTERACTS_WITH, INTERACTS_WITH]",Lisinopril
3,"[INTERACTS_WITH, INTERACTS_WITH]",Enalapril
4,"[INTERACTS_WITH, INTERACTS_WITH]",Losartan


## `CREATE` vs `MERGE` vs `MATCH`

| Keyword | Behavior | Use when |
|---|---|---|
| `MATCH` | Read-only — fails silently if nothing matches | Querying |
| `CREATE` | Always creates a new node/edge — duplicates allowed | You're sure it doesn't exist |
| `MERGE` | "Match or Create" — idempotent | Ingestion / upserts |

**Rule of thumb:** in production ingestion (like NB01's loader), always use `MERGE`. It's idempotent — re-running won't create duplicates.

In [37]:
# CREATE — make a temporary test node (we'll delete it later)
run("CREATE (t:TestNode {name: 'temp_demo', created_by: 'cypher_fundamentals'}) RETURN t.name")

,t.name
0,temp_demo


In [40]:
# MERGE — runs once, creates it. Run twice, second run is a no-op.
run("MERGE (t:TestNode {name: 'merge_demo'}) RETURN t.name")

,t.name
0,merge_demo


In [41]:
# Run MERGE again — same row, no duplicate
run("MERGE (t:TestNode {name: 'merge_demo'}) RETURN t.name, count(*) AS rows")

,t.name,rows
0,merge_demo,1


In [42]:
# Verify only ONE merge_demo exists
run("MATCH (t:TestNode {name: 'merge_demo'}) RETURN count(t) AS count")

,count
0,1


### `MERGE` with `ON CREATE` / `ON MATCH`

You can attach extra clauses to do different things depending on whether the node was created or already existed.

In [44]:
run("""
MERGE (t:TestNode {name: 'upsert_demo'})
ON CREATE SET t.created_at = timestamp(), t.runs = 1
ON MATCH  SET t.runs = t.runs + 1
RETURN t.name, t.runs
""")

,t.name,t.runs
0,upsert_demo,2


## `SET`, `REMOVE`, `DELETE`, `DETACH DELETE`

| Keyword | What it does |
|---|---|
| `SET` | Add or update a property or label |
| `REMOVE` | Drop a property or label |
| `DELETE` | Delete a node — **fails if it has relationships** |
| `DETACH DELETE` | Delete a node *and* all its relationships |

In [45]:
# SET — add a property
run("MATCH (t:TestNode {name: 'merge_demo'}) SET t.note = 'hello' RETURN t.name, t.note")

,t.name,t.note
0,merge_demo,hello


In [46]:
# REMOVE — drop the property
run("MATCH (t:TestNode {name: 'merge_demo'}) REMOVE t.note RETURN t.name, t.note")

,t.name,t.note
0,merge_demo,None


In [47]:
# Cleanup — DETACH DELETE removes nodes even if they have relationships
run("MATCH (t:TestNode) DETACH DELETE t")
run("MATCH (t:TestNode) RETURN count(t) AS remaining")

,remaining
0,0


## `UNWIND` — turning a list into rows

`UNWIND` is the inverse of `collect()`. Give it a list, get one row per element. This is the foundation of efficient batch ingestion.

In [48]:
run("""
UNWIND ['Aspirin', 'Ibuprofen', 'Warfarin'] AS drug_name
MATCH (d:Drug {name: drug_name})
RETURN d.name, d.drug_class
""")

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `drug_class` does not exist in database `c621105e`. Verify that the spelling is correct.', position=<SummaryInputPosition line=4, column=18, offset=108>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 108, 'line': 4, 'column': 18}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nUNWIND ['Aspirin', 'Ibuprofen', 'Warfarin'] AS drug_name\nMATCH (d:Drug {name: drug_name})\nRETURN d.name, d.drug_class\n"


,d.name,d.drug_class
0,Aspirin,None
1,Ibuprofen,None
2,Warfarin,None


In [49]:
# UNWIND a list of dicts — typical batch ingestion pattern
run("""
UNWIND [
  {name: 'TestDrug1', cls: 'TestClass'},
  {name: 'TestDrug2', cls: 'TestClass'}
] AS row
MERGE (d:Drug {name: row.name})
ON CREATE SET d.drug_class = row.cls, d.is_test = true
RETURN d.name, d.is_test
""")

,d.name,d.is_test
0,TestDrug1,True
1,TestDrug2,True


In [50]:
# Cleanup test drugs
run("MATCH (d:Drug) WHERE d.is_test = true DETACH DELETE d")
run("MATCH (d:Drug) WHERE d.is_test = true RETURN count(d) AS remaining")

,remaining
0,0


## `UNION` and `UNION ALL`

Combine results from two queries, like SQL.

- `UNION` — removes duplicates
- `UNION ALL` — keeps duplicates

Both queries must return the **same column names**.

In [51]:
# All conditions associated with Aspirin — either treated OR contraindicated
run("""
MATCH (d:Drug {name: 'Aspirin'})-[:TREATS]->(c:Condition)
RETURN c.name AS condition, 'treats' AS rel
UNION
MATCH (d:Drug {name: 'Aspirin'})-[:CONTRAINDICATED_FOR]->(c:Condition)
RETURN c.name AS condition, 'contraindicated' AS rel
""")

,condition,rel
0,Pain,treats
1,Inflammation,treats
2,Fever,treats
3,Cardiovascular Disease Prevention,treats
4,Stroke Prevention,treats
5,Active GI Bleeding,contraindicated
6,Aspirin-Sensitive Asthma,contraindicated
7,Children with Viral Illness (Reye Syndrome Risk),contraindicated
8,Third Trimester Pregnancy,contraindicated


## List comprehensions in Cypher

Cypher has list comprehensions just like Python: `[item IN list WHERE pred | expression]`.

Most useful when working with paths.

In [52]:
# Path with mapped node names
run("""
MATCH path = (a:Drug {name: 'Aspirin'})-[*1..2]-(b:Drug)
WHERE a <> b
RETURN [n IN nodes(path) | n.name] AS chain,
       [r IN relationships(path) | type(r)] AS rels
LIMIT 5
""")

,chain,rels
0,"[Aspirin, Ibuprofen]",[INTERACTS_WITH]
1,"[Aspirin, Ibuprofen, Enoxaparin]","[INTERACTS_WITH, INTERACTS_WITH]"
2,"[Aspirin, Ibuprofen, Lisinopril]","[INTERACTS_WITH, INTERACTS_WITH]"
3,"[Aspirin, Ibuprofen, Enalapril]","[INTERACTS_WITH, INTERACTS_WITH]"
4,"[Aspirin, Ibuprofen, Losartan]","[INTERACTS_WITH, INTERACTS_WITH]"


In [53]:
# Conditional projection — only keep node names that aren't drugs
run("""
MATCH path = (a:Drug {name: 'Aspirin'})-[*1..2]-(b)
RETURN [n IN nodes(path) WHERE NOT 'Drug' IN labels(n) | n.name] AS non_drug_nodes
LIMIT 5
""")

,non_drug_nodes
0,[]
1,[]
2,[]
3,[]
4,[]


## Diamond / triangle patterns

These are the *signature* graph queries that SQL struggles with. The idea: **two entities sharing a common neighbor**.

```
    (Aspirin) ──TREATS──▶ (Pain) ◀──TREATS── (Ibuprofen)
```

This is a 'diamond' (two arrows meeting). It says: "these two drugs treat the same condition".

In [54]:
# Drug pairs that treat the SAME condition
run("""
MATCH (d1:Drug)-[:TREATS]->(c:Condition)<-[:TREATS]-(d2:Drug)
WHERE d1.name < d2.name      // avoid duplicate (a,b) and (b,a)
RETURN d1.name AS drug_a, d2.name AS drug_b, c.name AS shared_condition
ORDER BY shared_condition, drug_a
LIMIT 10
""")

,drug_a,drug_b,shared_condition
0,Clopidogrel,Prasugrel,Acute Coronary Syndrome
1,Clopidogrel,Heparin,Acute Coronary Syndrome
2,Clopidogrel,Ticagrelor,Acute Coronary Syndrome
3,Clopidogrel,Enoxaparin,Acute Coronary Syndrome
4,Enoxaparin,Prasugrel,Acute Coronary Syndrome
5,Enoxaparin,Heparin,Acute Coronary Syndrome
6,Enoxaparin,Ticagrelor,Acute Coronary Syndrome
7,Heparin,Ticagrelor,Acute Coronary Syndrome
8,Heparin,Prasugrel,Acute Coronary Syndrome
9,Prasugrel,Ticagrelor,Acute Coronary Syndrome


In [55]:
# Drugs that share a side effect with Aspirin
run("""
MATCH (a:Drug {name: 'Aspirin'})-[:CAUSES_SIDE_EFFECT]->(s:SideEffect)<-[:CAUSES_SIDE_EFFECT]-(other:Drug)
WHERE other <> a
RETURN other.name AS drug, collect(s.name) AS shared_side_effects
ORDER BY drug
""")

,drug,shared_side_effects
0,Amoxicillin,[Nausea]
1,Apixaban,"[Bruising, Nausea]"
2,Azithromycin,[Nausea]
3,Ciprofloxacin,[Nausea]
4,Citalopram,[Nausea]
5,Clopidogrel,[Bruising]
6,Codeine,[Nausea]
7,Dabigatran,[Nausea]
8,Diclofenac,[GI Bleeding]
9,Digoxin,[Nausea]


In [56]:
# Triangle: drug -> condition -> contraindicated drug
# 'is there a drug treating something Aspirin is contraindicated for?'
run("""
MATCH (a:Drug {name: 'Aspirin'})-[:CONTRAINDICATED_FOR]->(c:Condition)<-[:TREATS]-(other:Drug)
RETURN c.name AS condition, other.name AS treated_by
""")

""
